In [ ]:
%%capture
# see comments in README on changes to the conda venv
import os
# import modin.pandas as pd
import pandas as pd
from dj_notebook import activate
from pathlib import Path
env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option('future.no_silent_downcasting', True)


In this notebook split and flatten the reasons_ineligible field, delimited by pipe, into columns
Two filed will be exported:
1. reasons_ineligible.csv
2. reasons_ineligible_summary.csv

In [ ]:
from intecomm_analytics.dataframes import get_screening_df


In [ ]:
df = get_screening_df()


In [ ]:
# convert data to a dictionary of {screening_identifier: [reason, ...]}
dct = df[df.eligible ==0][["screening_identifier", "reasons_ineligible"]].to_dict()
reasons = {}
for index, screening_identifier in dct.get("screening_identifier").items():
    reasons.update({screening_identifier: list(set(dct.get("reasons_ineligible")[index].split("|")))})

# get reasons a unique list for dataframe columns
columns = []
for l in reasons.values():
    columns.extend(l)
columns = list(set(columns))
columns.sort()
columns.insert(0, "screening_identifier")

# build an empty dictionary for rows
row = {}
for col in columns:
    row.update({col:0})

# reasons per ineligible subject
data = []
for index, screening_identifier in dct.get("screening_identifier").items():
    r = row.copy()
    for reason in dct.get("reasons_ineligible")[index].split("|"):
        r[reason] = 1
        r["screening_identifier"] = screening_identifier
    data.append(r)
# build and export dataframe
df_reasons_ineligible = pd.DataFrame(data)
df_reasons_ineligible.to_csv(Path("/Users/erikvw/Documents/ucl/protocols/intecomm/analysis/primary/") / "reasons_ineligible.csv", index=False)

# create a summary of the above
data_sum = {}
for col in columns:
    if col == "screening_identifier":
        continue
    data_sum.update({col: [df_reasons_ineligible[col].sum()]})
# build and export dataframe
df_summary = pd.DataFrame(data_sum)
df_summary.melt(var_name='Reason', value_name='Count').to_csv(Path("/Users/erikvw/Documents/ucl/protocols/intecomm/analysis/primary/") / "reasons_ineligible_summary.csv", index=False)

In [ ]:
df[df.eligible==0][["screening_identifier", "reasons_ineligible"]].reasons_ineligible.value_counts(dropna=False)

In [ ]:
df[df.eligible==1][["screening_identifier", "reasons_ineligible"]].reasons_ineligible.value_counts(dropna=False)


In [ ]:
df[(df.eligible==1) & ~(df.subject_identifier.str.startswith("107-"))][["screening_identifier", "subject_identifier"]]


In [ ]:
df[(df.eligible==1) & (df.subject_identifier.str.startswith("107-")) & (df.group_identifier.isna())].groupby("site_id").size()


In [ ]:
df[(df.eligible==1) & (df.subject_identifier.str.startswith("107-")) & (df.group_identifier.isna())].groupby(["hiv_dx", "dm_dx", "htn_dx"]).size()

In [ ]:
DM, HTN or both = 77
HIV only = 14
Multi = 13

In [ ]:
23 + 12 + 42

In [ ]:
77+14+13